# House Price Dataset Exploration

## Objective:
The objective of this project is to explore the factors influencing residential property prices and build a multiple linear regression model capable of predicting house prices based on key structural characteristics and amenities.

### Loading Libraries & Dataset

In [6]:
import pandas as pd
housing = pd.read_csv("/Users/poojachandna/Documents/data science portfolio/housing price predictor/data/housing_data.csv")
housing.head()

,price,area,bedrooms,hot water,airconditioning
0,13300000,7420,4,no,yes
1,12250000,8960,4,no,yes
2,12250000,9960,3,no,no
3,12215000,7500,4,no,yes
4,11410000,7420,4,no,yes


### Dataset Dimensions

In [8]:
housing.shape

(545, 5)

### Dataset Information

In [9]:
housing.info()

<class 'pandas.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   price            545 non-null    int64
 1   area             545 non-null    int64
 2   bedrooms         545 non-null    int64
 3   hot water        545 non-null    str  
 4   airconditioning  545 non-null    str  
dtypes: int64(3), str(2)
memory usage: 23.7 KB


### Checking for Missing values

In [10]:
housing.isnull().sum()

price              0
area               0
bedrooms           0
hot water          0
airconditioning    0
dtype: int64

### Summary Statistics

In [11]:
housing.describe()

,price,area,bedrooms
count,5.450000e+02,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138
std,1.870440e+06,2170.141023,0.738064
min,1.750000e+06,1650.000000,1.000000
25%,3.430000e+06,3600.000000,2.000000
50%,4.340000e+06,4600.000000,3.000000
75%,5.740000e+06,6360.000000,3.000000
max,1.330000e+07,16200.000000,6.000000


In [12]:
import plotly.express as px

### 1. Distribution of House Prices

In [21]:
fig = px.histogram(housing, x="price", title="Distribution of Housing Prices", color_discrete_sequence=["#B8E2F2"])
fig.show()

The distribution of house prices appears Right-Skewed, with most clustered around lower prices and only a few high-value properties. These high-value properties thus contribute in increasing the average price.

### 2. Area vs Price

In [ ]:
fig = px.scatter(
    housing,
    x="area",
    y="price",
    title="Relationship Between House Area and Price",
    trendline="ols",
 color_discrete_sequence=["#8FD3FF"])

#changing the color of the regression line to make it stand out
fig.data[1].update(
    line=dict(
        color="#0B3D91",
        width=2))

fig.show()

The scatter plot indicates a positive relation between House Area and Price, suggesting that larger houses usually attract higher prices. However, given the moderate dispersion of data points around the regression line, we know that area alone does not explain the variations in property values.

In [26]:
housing["area"].corr(housing["price"])

np.float64(0.5359973457780803)

The Pearson correlation coefficient between house area and price is 0.53, indicating a moderate positive linear relationship. This numerical result supports the visual interpretation of the scatter plot: larger houses tend to be more expensive, but other factors also contribute to price variation.

### 3. No. of Bedrooms vs Price

In [ ]:
fig = px.box(
    housing,
    x="bedrooms",
    y="price",
    title="House Prices by Number of Bedrooms",
    color_discrete_sequence=["#B8E2F2"])

fig.show()

The median house price generally increases as the number of bedrooms increases, suggesting a positive association between the number of bedrooms and property value. However, price variability also becomes larger for houses with more bedrooms, indicating that bedroom count alone does not fully explain differences in house prices. Several high-priced outliers are present, particularly among 3-bedroom and 4-bedroom houses, suggesting that additional property characteristics influence market value.

In [28]:
housing["bedrooms"].corr(housing["price"])

np.float64(0.3664940257738685)

The Pearson correlation coefficient between the number of bedrooms and house price indicates a weak positive linear relationship. The positive correlation suggests that houses with a greater number of bedrooms generally tend to have higher prices. However, the relatively weak strength of the relationship indicates that bedroom count alone is not a strong predictor of house prices.

### 4. Water Heating vs Price

In [31]:
fig = px.box(
    housing,
    x="hot water",
    y="price",
    title="House Prices with and without Hot Water Heating", color_discrete_sequence=["#B8E2F2"])

fig.show()

The box plot indicates that houses with hot water heating generally have a higher median selling price than those without hot water heating. However, there is substantial overlap between the two distributions, suggesting that the presence of hot water heating alone does not determine house prices. Several high-priced outliers are observed among houses without hot water heating, indicating that other property characteristics also play an important role in influencing market value.

### 5. Air Conditioning vs Price

In [32]:
fig = px.box(
    housing,
    x="airconditioning",
    y="price",
    title="House Prices with and without Air Conditioning", color_discrete_sequence=["#B8E2F2"])

fig.show()

Houses equipped with air conditioning generally have higher selling prices, as indicated by the higher median and upward shift in the overall price distribution. However, there is still considerable overlap between the two groups, suggesting that air conditioning alone does not determine property value. Several high-priced outliers are observed among houses without air conditioning, indicating that other characteristics, such as house area or additional amenities, may have a stronger influence on selling price.

## Data Pre-processing

### Converting yes/no Columns to Dummy Variables

In [36]:
housing_encoded = housing.copy()

housing_encoded["hot water"] = housing_encoded["hot water"].map({"yes": 1, "no": 0})
housing_encoded["airconditioning"] = housing_encoded["airconditioning"].map({"yes": 1, "no": 0})

housing_encoded.head()

,price,area,bedrooms,hot water,airconditioning
0,13300000,7420,4,0,1
1,12250000,8960,4,0,1
2,12250000,9960,3,0,0
3,12215000,7500,4,0,1
4,11410000,7420,4,0,1


### Finding the Variable Correlations

In [37]:
corr_matrix = housing_encoded.corr()
corr_matrix

,price,area,bedrooms,hot water,airconditioning
price,1.000000,0.535997,0.366494,0.093073,0.452954
area,0.535997,1.000000,0.151858,-0.009229,0.222393
bedrooms,0.366494,0.151858,1.000000,0.046049,0.160603
hot water,0.093073,-0.009229,0.046049,1.000000,-0.130023
airconditioning,0.452954,0.222393,0.160603,-0.130023,1.000000


In [40]:
fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="Blues",
    title="Correlation Matrix")

fig.show()


The correlation matrix indicates that house area has the strongest positive relationship with selling price (r = 0.54), followed by air conditioning (r = 0.45) and the number of bedrooms (r = 0.37). In contrast, hot water heating exhibits only a very weak positive correlation with price (r = 0.09), suggesting that it has little independent association with house prices.

The correlations among the independent variables are relatively low, indicating limited linear relationships between the predictors. This suggests that the variables provide distinct information and are unlikely to suffer from severe multicollinearity, making them suitable candidates for inclusion in a multiple regression model.

## Multiple Linear Regression

### Import statsmodels

In [43]:
import statsmodels.api as sm

### Define Dependent (Y) and Independent (X) Variables

In [41]:
X = housing_encoded[["area", "bedrooms", "hot water", "airconditioning"]]

y = housing_encoded["price"]

### Add constant

In [44]:
X = sm.add_constant(X)

### Fitting the Model

In [46]:
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.481
Model:                            OLS   Adj. R-squared:                  0.477
Method:                 Least Squares   F-statistic:                     125.2
Date:                Tue, 14 Jul 2026   Prob (F-statistic):           1.48e-75
Time:                        01:13:45   Log-Likelihood:                -8464.7
No. Observations:                 545   AIC:                         1.694e+04
Df Residuals:                     540   BIC:                         1.696e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            5.797e+05   2.62e+05     

### Regression Results

The multiple linear regression model is statistically significant (F-statistic p < 0.001), indicating that the selected predictors collectively explain a significant proportion of the variation in house prices.

The model achieved an R² value of *0.481*, suggesting that approximately *48.1%* of the variation in house prices is explained by house area, number of bedrooms, hot water heating, and air conditioning.

All four predictors were statistically significant (p < 0.001). House area exhibited a positive relationship with price, with each additional unit of area increasing the predicted selling price by approximately *₹367*, holding other variables constant. Similarly, an additional bedroom increased the predicted price by approximately *₹613,000*. Houses with hot water heating and air conditioning were associated with price premiums of approximately *₹1.16 million* and *₹1.35 million*, respectively.

These findings suggest that both structural characteristics (area and bedrooms) and amenities (hot water heating and air conditioning) contribute significantly to explaining house prices in this dataset.

## Model Testing

### Generating Predictions

In [48]:
housing_encoded["predicted_price"] = model.predict(X)
housing_encoded.head()

,price,area,bedrooms,hot water,airconditioning,predicted_price
0,13300000,7420,4,0,1,7.106052e+06
1,12250000,8960,4,0,1,7.671059e+06
2,12250000,9960,3,0,0,6.072784e+06
3,12215000,7500,4,0,1,7.135403e+06
4,11410000,7420,4,0,1,7.106052e+06


### Calculating Residuals (Residual = Actual Price - Predicted Price)

In [50]:
housing_encoded["residuals"] = (housing_encoded["price"] - housing_encoded["predicted_price"])
housing_encoded.head()

,price,area,bedrooms,hot water,airconditioning,predicted_price,residuals
0,13300000,7420,4,0,1,7.106052e+06,6.193948e+06
1,12250000,8960,4,0,1,7.671059e+06,4.578941e+06
2,12250000,9960,3,0,0,6.072784e+06,6.177216e+06
3,12215000,7500,4,0,1,7.135403e+06,5.079597e+06
4,11410000,7420,4,0,1,7.106052e+06,4.303948e+06


### Actual vs Predicted

In [51]:
fig = px.scatter(
    housing_encoded,
    x="price",
    y="predicted_price",
    title="Actual vs Predicted House Prices",
    labels={
        "price":"Actual Price",
        "predicted_price":"Predicted Price"
    },
    color_discrete_sequence=["#8FD3FF"]
)

fig.add_shape(
    type="line",
    x0=housing_encoded["price"].min(),
    y0=housing_encoded["price"].min(),
    x1=housing_encoded["price"].max(),
    y1=housing_encoded["price"].max(),
    line=dict(
        color="red",
        dash="dash"
    )
)

fig.show()

The actual versus predicted plot demonstrates a clear positive relationship between observed and predicted house prices, indicating that the regression model captures the overall pricing trend. Most mid-priced properties are predicted reasonably well, with observations lying relatively close to the line of perfect prediction.

However, the model consistently underestimates several high-value properties, suggesting that the current set of predictors does not fully explain premium house prices. This indicates that additional variables, such as location or property quality, may be required to improve predictive performance.

### Residual Plot

In [52]:
fig = px.scatter(
    housing_encoded,
    x="predicted_price",
    y="residuals",
    title="Residuals vs Predicted Values",
    labels={
        "predicted_price":"Predicted Price",
        "residuals":"Residual"
    },
    color_discrete_sequence=["#8FD3FF"]
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="red"
)

fig.show()

The residuals are randomly distributed around the zero line, indicating that the regression model does not exhibit a clear systematic bias. No obvious nonlinear pattern is observed, suggesting that a linear model is appropriate for this dataset.

However, the spread of residuals increases slightly for higher predicted prices, indicating that the model makes larger prediction errors for more expensive houses. This suggests the presence of mild heteroscedasticity and indicates that additional variables may improve the model's predictive performance.

### Residual Distribution

In [53]:
fig = px.histogram(
    housing_encoded,
    x="residuals",
    title="Distribution of Residuals",
    color_discrete_sequence=["#8FD3FF"]
)

fig.show()

The residuals are approximately centered around zero and exhibit a roughly bell-shaped distribution, indicating that the regression model satisfies the normality assumption reasonably well. However, the distribution displays a slight positive skew, with a longer right tail caused by a small number of houses that are substantially underpredicted.

Overall, the residual distribution suggests that the linear regression model provides an adequate fit for the majority of observations, while indicating room for improvement in predicting high-value properties.

### Calculating RMSE

In [54]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(
    mean_squared_error(
        housing_encoded["price"],
        housing_encoded["predicted_price"]
    )
)

print(f"RMSE: ₹{rmse:,.2f}")

RMSE: ₹1,346,010.74


The model achieved a Root Mean Squared Error of *₹1,346,010.74*, indicating that the predicted house prices differ from the actual selling prices by approximately *₹1.35 million on average*.

Considering the simplicity of the model and the limited number of predictors, this represents a reasonable level of prediction error. The remaining error is likely explained by additional property characteristics, such as location, construction quality, and other amenities, which are not included in the dataset.

## Key Findings


- **House area is the strongest predictor of house prices.** It exhibited the highest positive correlation with price (r = 0.54), indicating that larger houses generally command higher market values.

- **Air conditioning and hot water heating contribute significantly to property value.** The regression model estimated that houses with air conditioning and hot water heating were associated with average price premiums of approximately ₹1.35 million and ₹1.16 million, respectively, after controlling for other variables.

- **The number of bedrooms positively influences house prices**, although its relationship with price is weaker than that of house area. This suggests that total living space is a more informative predictor than bedroom count alone.

- **The multiple linear regression model was statistically significant** (p < 0.001) and explained approximately **48.1% of the variation** in house prices (R² = 0.481), indicating that the selected variables capture a substantial portion of the pricing differences.

- **Model diagnostics indicated an acceptable overall fit.** Residuals were centered around zero with no strong systematic patterns, although prediction errors increased slightly for higher-priced houses.

- **The model achieved an RMSE of approximately ₹1.35 million**, meaning that predicted prices differed from actual selling prices by about ₹1.35 million on average.

- **The remaining unexplained variation is likely due to omitted factors**, such as property location, construction quality, furnishing, parking availability, neighborhood characteristics, and other amenities that were not included in the dataset.